# Curated EEGMMIDB MI Fine-Tuning with SignalJEPA (Within-Subject 5-Fold)

Configurable notebook for the curated PhysioNet EEGMMIDB / PhysioNet 2009 motor imagery dataset. It follows the same organization as `moabb_mi_sjepa`: average reference, resample to 128 Hz, bandpass 0.5-40 Hz, fixed event-window creation, post-window microvolt scaling, S-JEPA downstream model loading, and artifact saving.

The main difference from the MOABB notebook is that the curated dataset is loaded from CSV files first, converted into MNE `Raw` objects, preprocessed, and saved as FIF files so the next run can reuse the loaded/preprocessed data without reparsing every CSV.

# 1. Setup

In [ ]:
import os
import random
import sys
from pathlib import Path
import platform
import re
import json
import hashlib
from datetime import datetime
import math
import shutil

import numpy as np
import pandas as pd

import torch
from torch.utils.data import TensorDataset, Subset

from braindecode import EEGClassifier
from braindecode.models import SignalJEPA_PreLocal, SignalJEPA_PostLocal, SignalJEPA_Contextual

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from skorch.callbacks import EarlyStopping
from skorch.dataset import ValidSplit

import builtins

import mne
from mne.channels import make_standard_montage
mne.set_log_level("WARNING")

print("All imports loaded successfully.")

In [ ]:
print("Runtime Environment:")
print(f"  - Python: {sys.version}")
print(f"  - Platform: {platform.platform()}")

WORKING_DIR = Path.cwd().parent
NOTEBOOK_DIR = Path.cwd()
print(f"\nWorking directory: {WORKING_DIR}")
print(f"Notebook directory: {NOTEBOOK_DIR}")

# 2. Configuration

## 2.1 Default Dataset Settings

In [ ]:
# Defaults for the curated PhysioNet EEGMMIDB motor-imagery transfer experiment.
# Standard PhysioNet imagined left/right fist runs are 04, 08, and 12.
CURATED_EEGMMIDB_CONFIGS = {
    "Curated_EEGMMIDB": {
        "goal": "curated PhysioNet EEGMMIDB 4-second MI transfer test",
        "labels_to_keep": ["left_hand", "right_hand"],
        "exclude_subjects": [],
        "default_subjects": None,  # None = all subjects found in the curated CSV directory
        "mi_runs": ["04", "08", "12"],
        "fallback_mi_runs_if_empty": ["02", "06", "10"],
        "base_trial_duration_s": 4.0,
        "target_window_duration_s": 4.2,
        "original_sfreq": 160,
        "raw_label_to_name": {5: "left_hand", 6: "right_hand"},
    },
}

## 2.2 CONFIG

In [ ]:
CONFIG = {
    # Paths
    "artifact_dir": str(WORKING_DIR / "artifacts" / "curated-eegmmidb-sjepa"),

    # Dataset source
    # - "curated": load curated CSV files, preprocess, and save/reuse a FIF cache
    # - "preprocessed": load an already saved FIF cache from preprocessed_dir
    "dataset_source": "curated",
    "dataset_name": "Curated_EEGMMIDB",
    "dataset_path": str(NOTEBOOK_DIR / "_datasets" / "eegmmidb"),
    "preprocessed_dir": None,

    # Set these to None to use CURATED_EEGMMIDB_CONFIGS defaults.
    "labels_to_keep": None,
    "exclude_subjects": None,
    "subjects_to_use": None,
    "mi_runs": None,
    "fallback_mi_runs_if_empty": None,
    "raw_label_to_name": None,
    "target_window_duration_s": None,
    "original_sfreq": None,

    # Curated CSV format
    "file_pattern": r"SUB_(\d+)_(SIG|ANN)_(\d+)\.csv",
    "curated_signal_units": "microvolts",  # microvolts or volts

    # Preprocessing
    "sfreq": 128,
    "bandpass_low": 0.5,
    "bandpass_high": 40.0,
    "channel_montage": "standard_1020",

    # Model settings
    "model_name": "SignalJEPA_PreLocal",  # SignalJEPA_PreLocal, SignalJEPA_PostLocal, SignalJEPA_Contextual
    "pretrained_mode": "from_pretrained", # from_pretrained or random

    # Strategy settings
    "strategy": "new",  # new, full
    "warmup_epochs": 10,

    # Cross-validation
    "cv_folds": 5,
    "val_split": 0.0,

    # Training
    "batch_size": 32,
    "n_epochs": 50,
    "early_stopping_patience": None,
    "learning_rate": 0.001,

    # Reproducibility
    "seed": 12,
    "set_seed": True,
}

In [ ]:
if CONFIG["dataset_name"] not in CURATED_EEGMMIDB_CONFIGS:
    raise ValueError(
        f"Unsupported dataset_name={CONFIG['dataset_name']}. "
        f"Supported: {sorted(CURATED_EEGMMIDB_CONFIGS)}"
    )

_DATASET_DEFAULTS = CURATED_EEGMMIDB_CONFIGS[CONFIG["dataset_name"]]

# Fill defaults into CONFIG so config.json contains the effective run settings.
if CONFIG["labels_to_keep"] is None:
    CONFIG["labels_to_keep"] = list(_DATASET_DEFAULTS["labels_to_keep"])
if CONFIG["exclude_subjects"] is None:
    CONFIG["exclude_subjects"] = list(_DATASET_DEFAULTS["exclude_subjects"])
if CONFIG["mi_runs"] is None:
    CONFIG["mi_runs"] = list(_DATASET_DEFAULTS["mi_runs"])
if CONFIG["fallback_mi_runs_if_empty"] is None:
    CONFIG["fallback_mi_runs_if_empty"] = list(_DATASET_DEFAULTS["fallback_mi_runs_if_empty"])
if CONFIG["raw_label_to_name"] is None:
    CONFIG["raw_label_to_name"] = dict(_DATASET_DEFAULTS["raw_label_to_name"])
if CONFIG["target_window_duration_s"] is None:
    CONFIG["target_window_duration_s"] = float(_DATASET_DEFAULTS["target_window_duration_s"])
if CONFIG["original_sfreq"] is None:
    CONFIG["original_sfreq"] = float(_DATASET_DEFAULTS["original_sfreq"])

CLASSES_MAPPING = {label: idx for idx, label in enumerate(CONFIG["labels_to_keep"])}
TARGET_N_CLASSES = len(CLASSES_MAPPING)
BASE_TRIAL_DURATION_S = float(_DATASET_DEFAULTS["base_trial_duration_s"])
TARGET_TRIAL_DURATION_S = float(CONFIG["target_window_duration_s"])

print("Selected dataset defaults:")
print(f"  Dataset:                 {CONFIG['dataset_name']}")
print(f"  Goal:                    {_DATASET_DEFAULTS['goal']}")
print(f"  Labels:                  {CONFIG['labels_to_keep']}")
print(f"  Class mapping:           {CLASSES_MAPPING}")
print(f"  Excluded subjects:       {CONFIG['exclude_subjects']}")
print(f"  MI runs:                 {CONFIG['mi_runs']}")
print(f"  Fallback MI runs:        {CONFIG['fallback_mi_runs_if_empty']}")
print(f"  Target window duration:  {TARGET_TRIAL_DURATION_S}")

## 2.3 Artifact Creation and Logging Init

In [ ]:
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    config_str = json.dumps(CONFIG, sort_keys=True, default=str)
    config_hash = hashlib.md5(config_str.encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / CONFIG["dataset_name"] / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")

def _safe_write_text(stream, text):
    try:
        stream.write(text)
        return
    except UnicodeEncodeError:
        pass

    # Fallback for Windows terminals using a limited code page.
    encoding = getattr(stream, "encoding", None) or "utf-8"
    safe_text = text.encode(encoding, errors="replace").decode(encoding, errors="replace")
    stream.write(safe_text)

def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop("sep", " ")
    end = kwargs.pop("end", "\n")
    flush = kwargs.pop("flush", False)
    file = kwargs.pop("file", None)
    message = sep.join(str(arg) for arg in args)
    leading_newlines = len(message) - len(message.lstrip("\n"))
    message_body = message[leading_newlines:]

    def _write_target(text):
        if file is None:
            _safe_write_text(sys.stdout, text)
            if flush:
                sys.stdout.flush()
        else:
            _safe_write_text(file, text)
            if flush and hasattr(file, "flush"):
                file.flush()

    if leading_newlines > 0:
        blanks = "\n" * leading_newlines
        _write_target(blanks)
        _safe_write_text(_LOG_FILE_HANDLE, blanks)

    if message_body:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        stamped = f"[{ts}] {message_body}"
        _write_target(stamped + end)
        _safe_write_text(_LOG_FILE_HANDLE, stamped + end)
    else:
        _write_target(end)
        _safe_write_text(_LOG_FILE_HANDLE, end)

    if flush:
        _LOG_FILE_HANDLE.flush()

builtins.print = _timestamped_print

## 2.4 Reproducibility

In [ ]:
def resolve_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

DEVICE = resolve_device()
print(f"Using device: {DEVICE}")

def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

BASE_SEED = int(CONFIG["seed"]) if CONFIG["seed"] is not None else None
if CONFIG["set_seed"]:
    if BASE_SEED is None:
        raise ValueError("Seed control is enabled but CONFIG['seed'] is None.")
    seed_everything(BASE_SEED)
    print(f"Seed initialized: {BASE_SEED}")
else:
    print("Seed control disabled")

print("=" * 70)
print("CONFIGURATION")
print("=" * 70)
for key in sorted(CONFIG.keys()):
    print(f"  {key}: {CONFIG[key]}")
print("=" * 70)

config_path = ARTIFACT_DIR / "config.json"
with open(config_path, "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)
print(f"Config saved to: {config_path}")

# 3. Load and Prepare Data

## 3.1 Derived Values

In [ ]:
def compute_epoch_window(sfreq, target_trial_duration_s):
    sfreq = float(sfreq)
    target_trial_duration_s = float(target_trial_duration_s)
    window_size_samples = int(math.floor(target_trial_duration_s * sfreq))
    print("Epoch window configuration:")
    print(f"  Target sfreq:            {sfreq:.0f} Hz")
    print(f"  Requested duration:      {target_trial_duration_s:.3f} s")
    print(f"  Expected window samples: {window_size_samples}")
    return window_size_samples

WINDOW_SAMPLES = compute_epoch_window(CONFIG["sfreq"], TARGET_TRIAL_DURATION_S)

DATASET_PATH = Path(CONFIG["dataset_path"])
if CONFIG["dataset_source"] not in {"curated", "preprocessed"}:
    raise ValueError("CONFIG['dataset_source'] must be either 'curated' or 'preprocessed'.")

if CONFIG["dataset_source"] == "curated" and not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Curated EEGMMIDB dataset not found at {DATASET_PATH}. "
        "Update CONFIG['dataset_path'] to the folder containing SUB_*_SIG_*.csv and SUB_*_ANN_*.csv."
    )

print(f"Curated dataset path: {DATASET_PATH}")

## 3.2 Load Data

In [ ]:
def normalize_subject_id(subject_id):
    return f"{int(subject_id):03d}" if str(subject_id).isdigit() else str(subject_id)

def normalize_run_id(run_id):
    return f"{int(run_id):02d}" if str(run_id).isdigit() else str(run_id)

def _sort_subject_key(x):
    sx = str(x)
    return int(sx) if sx.isdigit() else sx

def index_eegmmidb_files(base_path):
    print("Indexing curated EEGMMIDB CSV files...")
    base_path = Path(base_path)
    file_pattern = re.compile(CONFIG["file_pattern"])
    sig_files = {}
    ann_files = {}

    for csv_file in base_path.glob("*.csv"):
        match = file_pattern.search(csv_file.name)
        if not match:
            continue
        subject_id = normalize_subject_id(match.group(1))
        file_type = match.group(2)
        run_id = normalize_run_id(match.group(3))
        key = (subject_id, run_id)
        if file_type == "SIG":
            sig_files[key] = csv_file
        elif file_type == "ANN":
            ann_files[key] = csv_file

    sig_keys = set(sig_files.keys())
    ann_keys = set(ann_files.keys())
    missing_ann = sig_keys - ann_keys
    missing_sig = ann_keys - sig_keys
    if missing_ann or missing_sig:
        print(f"Warning: missing SIG/ANN pairs: missing_ann={sorted(missing_ann)[:10]}, missing_sig={sorted(missing_sig)[:10]}")

    keys = sorted(sig_keys & ann_keys)
    return {key: {"sig_path": sig_files[key], "ann_path": ann_files[key]} for key in keys}

if CONFIG["dataset_source"] == "curated":
    FILE_INDEX = index_eegmmidb_files(DATASET_PATH)
    ALL_SUBJECTS_FOUND = sorted({s for s, _ in FILE_INDEX.keys()}, key=_sort_subject_key)
    ALL_RUNS_FOUND = sorted({r for _, r in FILE_INDEX.keys()})
    print(f"Found paired SIG/ANN files: {len(FILE_INDEX)}")
    print(f"Subjects found: {len(ALL_SUBJECTS_FOUND)} | first={ALL_SUBJECTS_FOUND[:5]}")
    print(f"Runs found: {ALL_RUNS_FOUND}")

    if CONFIG["subjects_to_use"] is None:
        default_subjects = _DATASET_DEFAULTS.get("default_subjects")
        SUBJECTS = ALL_SUBJECTS_FOUND if default_subjects is None else [normalize_subject_id(s) for s in default_subjects]
    else:
        SUBJECTS = [normalize_subject_id(s) for s in CONFIG["subjects_to_use"]]

    excluded = {normalize_subject_id(s) for s in CONFIG["exclude_subjects"]}
    SUBJECTS = [s for s in SUBJECTS if s not in excluded]
    RUN_IDS = [normalize_run_id(r) for r in CONFIG["mi_runs"]]
else:
    FILE_INDEX = None
    SUBJECTS = [normalize_subject_id(s) for s in CONFIG["subjects_to_use"]] if CONFIG["subjects_to_use"] is not None else None
    RUN_IDS = [normalize_run_id(r) for r in CONFIG["mi_runs"]]

label_fragment = "_".join(CONFIG["labels_to_keep"])
label_hash = hashlib.md5(label_fragment.encode()).hexdigest()[:8]
subject_fragment = "all" if SUBJECTS is None else "_".join(str(s) for s in SUBJECTS)
subject_hash = hashlib.md5(subject_fragment.encode()).hexdigest()[:8]
run_fragment = "_".join(RUN_IDS)
run_hash = hashlib.md5(run_fragment.encode()).hexdigest()[:6]
window_hash = hashlib.md5(str(TARGET_TRIAL_DURATION_S).encode()).hexdigest()[:6]

if CONFIG["preprocessed_dir"] is None:
    PREPROCESSED_DATASETS_NAME = (
        f"{CONFIG['dataset_name']}_{label_hash}_runs_{run_hash}_w{window_hash}_subjects_{subject_hash}"
    )
    PREPROCESSED_DIR = WORKING_DIR / "preprocessed_curated_datasets" / PREPROCESSED_DATASETS_NAME
else:
    PREPROCESSED_DIR = Path(CONFIG["preprocessed_dir"])

PREPROCESSED_DATASETS_EXISTS = (PREPROCESSED_DIR / "preprocessed_index.json").exists()
print(f"\nSelected subjects: {len(SUBJECTS) if SUBJECTS is not None else 'from cache'}")
if SUBJECTS is not None:
    print(f"Subject IDs: {SUBJECTS[:20]}{' ...' if len(SUBJECTS) > 20 else ''}")
print(f"Selected run IDs: {RUN_IDS}")
print(f"Preprocessed cache exists: {PREPROCESSED_DATASETS_EXISTS} at {PREPROCESSED_DIR}")

if CONFIG["dataset_source"] == "preprocessed" and not PREPROCESSED_DATASETS_EXISTS:
    raise FileNotFoundError("CONFIG['dataset_source'] is 'preprocessed' but preprocessed_dir does not contain preprocessed_index.json.")

## 3.3 Preprocess Data

In [ ]:
CHANNEL_NAMES = [
    "FC5", "FC3", "FC1", "FCZ", "FC2", "FC4", "FC6",
    "C5", "C3", "C1", "CZ", "C2", "C4", "C6",
    "CP5", "CP3", "CP1", "CPZ", "CP2", "CP4", "CP6",
    "FP1", "FPZ", "FP2",
    "AF7", "AF3", "AFZ", "AF4", "AF8",
    "F7", "F5", "F3", "F1", "FZ", "F2", "F4", "F6", "F8",
    "FT7", "FT8",
    "T7", "T8", "T9", "T10",
    "TP7", "TP8",
    "P7", "P5", "P3", "P1", "PZ", "P2", "P4", "P6", "P8",
    "PO7", "PO3", "POZ", "PO4", "PO8",
    "O1", "OZ", "O2", "IZ",
]

INFO = mne.create_info(
    ch_names=CHANNEL_NAMES,
    sfreq=float(CONFIG["original_sfreq"]),
    ch_types=["eeg"] * len(CHANNEL_NAMES),
)
try:
    montage = make_standard_montage(CONFIG["channel_montage"])
    INFO.set_montage(montage, on_missing="ignore")
except Exception as exc:
    print(f"Montage setup warning: {exc}")

CHS_INFO = INFO["chs"]
print(f"Created MNE Info with {len(CHANNEL_NAMES)} EEG channels at {CONFIG['original_sfreq']} Hz")

In [ ]:
def _signal_to_volts(sig):
    units = str(CONFIG["curated_signal_units"]).lower()
    if units in {"microvolts", "uv", "µv"}:
        return sig.astype(np.float32) * 1e-6
    if units in {"volts", "v"}:
        return sig.astype(np.float32)
    raise ValueError("CONFIG['curated_signal_units'] must be 'microvolts' or 'volts'.")

def _raw_label_to_description(raw_label):
    label_map = {int(k): v for k, v in CONFIG["raw_label_to_name"].items()}
    return label_map.get(int(raw_label))

def build_raw_for_run(sig_path, ann_path):
    sig = pd.read_csv(sig_path, header=None).values
    if sig.shape[1] != len(CHANNEL_NAMES):
        raise ValueError(f"Expected {len(CHANNEL_NAMES)} channels, got {sig.shape[1]} in {sig_path}")

    data_volts = _signal_to_volts(sig).T
    raw = mne.io.RawArray(data_volts, INFO.copy(), verbose=False)

    ann = pd.read_csv(ann_path, header=None).values
    onsets = []
    durations = []
    descriptions = []
    for row in ann:
        raw_label = int(row[0])
        desc = _raw_label_to_description(raw_label)
        if desc is None:
            continue
        # Old curated implementation used row[3] as 1-based start sample and row[4] as end sample.
        start_idx = int(row[3]) - 1
        end_idx = int(row[4])
        if start_idx < 0 or end_idx <= start_idx or end_idx > sig.shape[0]:
            continue
        onsets.append(start_idx / float(CONFIG["original_sfreq"]))
        durations.append((end_idx - start_idx) / float(CONFIG["original_sfreq"]))
        descriptions.append(desc)

    raw.set_annotations(mne.Annotations(onset=onsets, duration=durations, description=descriptions))
    return raw

def preprocess_raw(raw):
    # Keep preprocessing fixed to match the MOABB notebook: EEG only -> average reference -> resample -> bandpass.
    raw.pick_types(eeg=True, meg=False, stim=False)
    mne.set_eeg_reference(raw, ref_channels="average", copy=False, verbose=False)
    raw.resample(float(CONFIG["sfreq"]), verbose=False)
    raw.filter(l_freq=CONFIG["bandpass_low"], h_freq=CONFIG["bandpass_high"], verbose=False)
    return raw

def count_target_annotations(raw):
    descriptions = np.asarray(raw.annotations.description).astype(str)
    return int(np.isin(descriptions, CONFIG["labels_to_keep"]).sum())

def preprocess_and_save_cache(run_ids):
    PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    raw_dir = PREPROCESSED_DIR / "raw_fif"
    raw_dir.mkdir(parents=True, exist_ok=True)

    records = []
    total_target_events = 0
    missing_pairs = []

    for subject_id in SUBJECTS:
        for run_id in run_ids:
            key = (subject_id, run_id)
            if key not in FILE_INDEX:
                missing_pairs.append(key)
                continue
            paths = FILE_INDEX[key]
            raw = build_raw_for_run(paths["sig_path"], paths["ann_path"])
            raw = preprocess_raw(raw)
            target_events = count_target_annotations(raw)
            total_target_events += target_events

            fif_path = raw_dir / f"sub-{subject_id}_run-{run_id}_raw.fif"
            raw.save(fif_path, overwrite=True, verbose=False)
            records.append({
                "subject_id": subject_id,
                "run_id": run_id,
                "raw_fif_path": str(fif_path),
                "n_times": int(raw.n_times),
                "sfreq": float(raw.info["sfreq"]),
                "n_annotations": int(len(raw.annotations)),
                "n_target_annotations": int(target_events),
            })

    metadata = {
        "dataset_name": CONFIG["dataset_name"],
        "dataset_path": str(DATASET_PATH),
        "subjects": list(SUBJECTS),
        "run_ids": list(run_ids),
        "labels_to_keep": list(CONFIG["labels_to_keep"]),
        "class_mapping": dict(CLASSES_MAPPING),
        "sfreq": float(CONFIG["sfreq"]),
        "bandpass_low": float(CONFIG["bandpass_low"]),
        "bandpass_high": float(CONFIG["bandpass_high"]),
        "target_window_duration_s": float(TARGET_TRIAL_DURATION_S),
        "window_samples": int(WINDOW_SAMPLES),
        "curated_signal_units": CONFIG["curated_signal_units"],
        "records": records,
        "missing_pairs_preview": missing_pairs[:20],
        "n_missing_pairs": len(missing_pairs),
        "total_target_events": int(total_target_events),
    }

    index_path = PREPROCESSED_DIR / "preprocessed_index.json"
    with open(index_path, "w") as f:
        json.dump(metadata, f, indent=2, default=str)

    print(f"Saved preprocessed FIF cache to: {PREPROCESSED_DIR}")
    print(f"  Records saved:        {len(records)}")
    print(f"  Missing pairs:        {len(missing_pairs)}")
    print(f"  Target annotations:   {total_target_events}")
    return metadata

if CONFIG["dataset_source"] == "curated" and not PREPROCESSED_DATASETS_EXISTS:
    print("Applying preprocessing and saving preprocessed FIF cache...")
    CACHE_METADATA = preprocess_and_save_cache(RUN_IDS)

    if CACHE_METADATA["total_target_events"] == 0 and CONFIG.get("fallback_mi_runs_if_empty"):
        fallback_runs = [normalize_run_id(r) for r in CONFIG["fallback_mi_runs_if_empty"]]
        print("No target annotations found with primary MI runs. Rebuilding cache with fallback runs...")
        print(f"Fallback runs: {fallback_runs}")
        shutil.rmtree(PREPROCESSED_DIR, ignore_errors=True)
        RUN_IDS = fallback_runs
        CONFIG["mi_runs_used"] = list(RUN_IDS)
        CACHE_METADATA = preprocess_and_save_cache(RUN_IDS)
    else:
        CONFIG["mi_runs_used"] = list(RUN_IDS)
else:
    print("Using existing preprocessed FIF cache.")
    CONFIG["mi_runs_used"] = list(RUN_IDS)

## 3.4 Load the preprocessed data

In [ ]:
INDEX_PATH = PREPROCESSED_DIR / "preprocessed_index.json"
if not INDEX_PATH.exists():
    raise FileNotFoundError(f"Missing preprocessed index: {INDEX_PATH}")

with open(INDEX_PATH, "r") as f:
    CACHE_METADATA = json.load(f)

PREPROCESSED_RUNS = []
for record in CACHE_METADATA["records"]:
    raw = mne.io.read_raw_fif(record["raw_fif_path"], preload=True, verbose=False)
    PREPROCESSED_RUNS.append({
        "subject_id": str(record["subject_id"]),
        "run_id": str(record["run_id"]),
        "raw": raw,
        "record": record,
    })

if len(PREPROCESSED_RUNS) == 0:
    raise RuntimeError("No preprocessed runs were loaded from cache.")

CHS_INFO = PREPROCESSED_RUNS[0]["raw"].info["chs"]
first_raw = PREPROCESSED_RUNS[0]["raw"]
all_annotation_labels = sorted({
    str(desc)
    for item in PREPROCESSED_RUNS
    for desc in np.asarray(item["raw"].annotations.description).astype(str)
})
subject_ids_loaded = sorted({item["subject_id"] for item in PREPROCESSED_RUNS}, key=_sort_subject_key)
run_ids_loaded = sorted({item["run_id"] for item in PREPROCESSED_RUNS})

SUBJECTS = subject_ids_loaded
RUN_IDS = run_ids_loaded

print("\nPost-load raw validation")
print(f"  Preprocessed dir:     {PREPROCESSED_DIR}")
print(f"  Recordings:          {len(PREPROCESSED_RUNS)}")
print(f"  Subjects loaded:     {subject_ids_loaded}")
print(f"  Run IDs loaded:      {run_ids_loaded}")
print(f"  sfreq first rec:     {first_raw.info['sfreq']} Hz")
print(f"  EEG channel count:   {len(first_raw.ch_names)}")
print(f"  Labels present:      {all_annotation_labels}")
print(f"  Labels selected:     {CONFIG['labels_to_keep']}")

## 3.5 Create Windows

In [ ]:
def summarize_annotation_counts(preprocessed_runs):
    counts = {}
    for item in preprocessed_runs:
        descriptions = np.asarray(item["raw"].annotations.description).astype(str)
        for desc in descriptions:
            counts[desc] = counts.get(desc, 0) + 1
    return dict(sorted(counts.items()))

def extract_windows_from_run(raw, target_duration_s, mapping):
    sfreq = float(raw.info["sfreq"])
    X = []
    y = []
    meta = []
    descriptions = np.asarray(raw.annotations.description).astype(str)
    for onset, duration, desc in zip(raw.annotations.onset, raw.annotations.duration, descriptions):
        if desc not in mapping:
            continue
        start = int(round(float(onset) * sfreq))
        stop = start + WINDOW_SAMPLES
        if start < 0 or stop > raw.n_times:
            continue
        x = raw.get_data(start=start, stop=stop).astype(np.float32)
        # Post-window microvolt scaling to match the MOABB notebook behavior.
        x = x * 1e6
        X.append(x)
        y.append(mapping[desc])
        meta.append({
            "onset": float(onset),
            "original_duration": float(duration),
            "label": str(desc),
            "start_sample": int(start),
            "stop_sample": int(stop),
        })
    return X, y, meta

def build_windows_dataset(preprocessed_runs, target_duration_s, mapping):
    before_counts = summarize_annotation_counts(preprocessed_runs)
    print(f"Annotation counts before filtering: {before_counts}")

    total_before = sum(len(item["raw"].annotations) for item in preprocessed_runs)
    target_set = set(mapping.keys())
    total_after = sum(
        int(np.isin(np.asarray(item["raw"].annotations.description).astype(str), list(target_set)).sum())
        for item in preprocessed_runs
    )
    print(f"Filtered annotations: kept {total_after} / {total_before}")
    print(f"Updated annotation durations:       {total_after} virtual windows at {target_duration_s:.3f}s")

    subject_arrays = {}
    dropped_recordings = 0
    total_windows = 0

    for item in preprocessed_runs:
        raw = item["raw"]
        subject_id = item["subject_id"]
        run_id = item["run_id"]
        X_run, y_run, meta_run = extract_windows_from_run(raw, target_duration_s, mapping)
        if len(y_run) == 0:
            dropped_recordings += 1
            continue
        if subject_id not in subject_arrays:
            subject_arrays[subject_id] = {"X": [], "y": [], "metadata": []}
        subject_arrays[subject_id]["X"].extend(X_run)
        subject_arrays[subject_id]["y"].extend(y_run)
        for m in meta_run:
            m["subject_id"] = subject_id
            m["run_id"] = run_id
            subject_arrays[subject_id]["metadata"].append(m)
        total_windows += len(y_run)

    if dropped_recordings > 0:
        print(f"Dropped recordings with zero usable windows: {dropped_recordings}")
    if len(subject_arrays) == 0:
        raise RuntimeError("No usable windows were created. Check run IDs, label mapping, and target window duration.")

    for subject_id, arrays in subject_arrays.items():
        arrays["X"] = np.stack(arrays["X"], axis=0).astype(np.float32)
        arrays["y"] = np.asarray(arrays["y"], dtype=np.int64)

    print(f"Total windows created: {total_windows}")
    return subject_arrays

SUBJECT_ARRAYS = build_windows_dataset(
    PREPROCESSED_RUNS,
    target_duration_s=TARGET_TRIAL_DURATION_S,
    mapping=CLASSES_MAPPING,
)

In [ ]:
first_subject = sorted(SUBJECT_ARRAYS.keys(), key=_sort_subject_key)[0]
x0 = SUBJECT_ARRAYS[first_subject]["X"][0]
y0 = SUBJECT_ARRAYS[first_subject]["y"][0]
print(f"Window shape: {tuple(x0.shape)}")
print(f"Window sample length expected={WINDOW_SAMPLES}, got={x0.shape[-1]}")
print(f"Class mapping: {CLASSES_MAPPING}")
ALL_LABELS = np.concatenate([arrays["y"] for arrays in SUBJECT_ARRAYS.values()])
print(f"Global class counts: {np.bincount(ALL_LABELS, minlength=TARGET_N_CLASSES).tolist()}")
print(f"Chance level: {1.0 / TARGET_N_CLASSES:.2f}")

## 3.6 Create Subject Windows

In [ ]:
def make_subject_dataset(X, y):
    return TensorDataset(torch.as_tensor(X, dtype=torch.float32), torch.as_tensor(y, dtype=torch.long))

def get_subject_windows(subject_arrays):
    subject_windows = {}
    for subject_id in sorted(subject_arrays.keys(), key=_sort_subject_key):
        arrays = subject_arrays[subject_id]
        if len(arrays["y"]) == 0:
            continue
        subject_windows[subject_id] = make_subject_dataset(arrays["X"], arrays["y"])
    if len(subject_windows) == 0:
        raise RuntimeError("No subject windows were created.")
    return subject_windows

SUBJECT_WINDOWS = get_subject_windows(SUBJECT_ARRAYS)

def summarize_subject_windows(subject_windows, n_classes):
    print("Summarizing per-subject windows...")
    for subject_id in sorted(subject_windows, key=_sort_subject_key):
        ds = subject_windows[subject_id]
        y = np.asarray([int(ds[i][1]) for i in range(len(ds))], dtype=np.int64)
        counts = np.bincount(y, minlength=n_classes)
        x = ds[0][0]
        print(f"  Subject {subject_id}: n_windows={len(ds)}, window_shape={tuple(x.shape)}, class_counts={counts.tolist()}")

summarize_subject_windows(SUBJECT_WINDOWS, TARGET_N_CLASSES)

# 4. Model

## 4.1 Build Model

In [ ]:
MODEL_REGISTRY = {
    "SignalJEPA_PreLocal": SignalJEPA_PreLocal,
    "SignalJEPA_PostLocal": SignalJEPA_PostLocal,
    "SignalJEPA_Contextual": SignalJEPA_Contextual,
}

DEFAULT_PRETRAINED_REPOS = {
    "SignalJEPA_Contextual": "braindecode/signal-jepa",
    "SignalJEPA_PostLocal": "braindecode/signal-jepa_without-chans",
    "SignalJEPA_PreLocal": "braindecode/signal-jepa_without-chans",
}

def get_model_class(model_name):
    if model_name not in MODEL_REGISTRY:
        raise ValueError(f"Unsupported model_name {model_name}; supported={list(MODEL_REGISTRY)}")
    return MODEL_REGISTRY[model_name]

def build_model(model_name):
    model_cls = get_model_class(model_name)
    n_chans = len(CHS_INFO)
    n_times = WINDOW_SAMPLES
    mode = CONFIG["pretrained_mode"]
    repo_id = DEFAULT_PRETRAINED_REPOS[model_name]
    if mode == "from_pretrained":
        model = model_cls.from_pretrained(
            repo_id,
            n_chans=n_chans,
            n_times=n_times,
            n_outputs=TARGET_N_CLASSES,
            strict=False,
        )
        info = {"loading_path": "from_pretrained", "repo_id": repo_id, "mode": mode}
    elif mode == "random":
        model = model_cls(n_chans=n_chans, n_times=n_times, n_outputs=TARGET_N_CLASSES)
        info = {"loading_path": "random_initialization", "repo_id": None, "mode": mode}
    else:
        raise ValueError("CONFIG['pretrained_mode'] must be 'from_pretrained' or 'random'.")
    return model, info

PRETRAINED_CHECKPOINT_INFO = {}

def initialize_model(model_name):
    model, info = build_model(model_name)
    info["model_name"] = model_name
    return model, info

In [ ]:
# Verify once that the selected model builds.
test_model, test_info = build_model(CONFIG["model_name"])
print(f"{CONFIG['model_name']} instantiated successfully.")
print(f"  Loading mode:          {test_info['loading_path']}")
print(f"  Repo:                  {test_info.get('repo_id')}")
print(f"  Total parameters:      {sum(p.numel() for p in test_model.parameters()):,}")
print(f"  Trainable parameters:  {sum(p.numel() for p in test_model.parameters() if p.requires_grad):,}")
print(test_model)
del test_model

## 4.2 Trainable Parameter Phases

In [ ]:
NEW_LAYER_PREFIXES = {
    "SignalJEPA_PreLocal": ("spatial_conv.", "final_layer."),
    "SignalJEPA_PostLocal": ("final_layer.",),
    "SignalJEPA_Contextual": ("pos_encoder.", "final_layer."),
}

def count_total_and_trainable_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def get_new_layer_prefixes(model_name):
    if model_name not in NEW_LAYER_PREFIXES:
        raise ValueError(f"Unsupported model_name for trainable phase logic: {model_name}")
    return NEW_LAYER_PREFIXES[model_name]

def set_trainable_params_for_phase(model, model_name, phase):
    if phase not in ("new", "warmup", "full"):
        raise ValueError(f"Unsupported phase: {phase}")
    trainable_names = []
    if phase == "full":
        for _, param in model.named_parameters():
            param.requires_grad = True
        trainable_names = [name for name, p in model.named_parameters() if p.requires_grad]
        phase_groups = ["all_parameters"]
    else:
        downstream_prefixes = get_new_layer_prefixes(model_name)
        for _, param in model.named_parameters():
            param.requires_grad = False
        for name, param in model.named_parameters():
            if any(name.startswith(prefix) for prefix in downstream_prefixes):
                param.requires_grad = True
                trainable_names.append(name)
        phase_groups = list(downstream_prefixes)

    total, trainable = count_total_and_trainable_params(model)
    if trainable == 0:
        raise RuntimeError(f"No trainable parameters found for model={model_name}, phase={phase}.")
    return {
        "model_name": model_name,
        "phase": phase,
        "trainable_groups": phase_groups,
        "total_params": int(total),
        "trainable_params": int(trainable),
        "trainable_ratio": float(trainable / total),
        "trainable_names": trainable_names,
    }

def assert_expected_trainable_scope(summary, model_name, phase):
    if phase == "full":
        return
    allowed_prefixes = get_new_layer_prefixes(model_name)
    unexpected_names = [name for name in summary["trainable_names"] if not any(name.startswith(prefix) for prefix in allowed_prefixes)]
    if unexpected_names:
        raise RuntimeError(f"Unexpected trainable parameters for {model_name} phase={phase}: {unexpected_names[:10]}")

def describe_trainable_params(summary, max_names=12):
    print(f"      Trainable groups: {summary['trainable_groups']}")
    print(f"      Total params:     {summary['total_params']:,}")
    print(f"      Trainable params: {summary['trainable_params']:,}")
    print(f"      Trainable pct:    {100.0 * summary['trainable_ratio']:.2f}%")
    if len(summary["trainable_names"]) <= max_names:
        preview_names = summary["trainable_names"]
    else:
        preview_names = summary["trainable_names"][:max_names]
    print(f"      Trainable parameter names: {preview_names}")
    if len(preview_names) < len(summary["trainable_names"]):
        print(f"      ... {len(summary['trainable_names']) - len(preview_names)} additional trainable parameters hidden")

# 5. Training

## 5.1 Build Classifier

In [ ]:
def get_targets(dataset):
    return np.asarray([int(dataset[i][1]) for i in range(len(dataset))], dtype=np.int64)

def make_train_split():
    val_split = CONFIG["val_split"]
    if val_split is None or float(val_split) <= 0.0:
        return None
    return ValidSplit(cv=float(val_split), stratified=True, random_state=12) # type: ignore

def make_callbacks():
    train_split = make_train_split()
    patience = CONFIG["early_stopping_patience"]
    if train_split is None or patience is None or int(patience) <= 0:
        return []

    return [
        (
            "early_stopping",
            EarlyStopping(
                monitor="valid_loss",
                patience=int(patience),
                lower_is_better=True,
                load_best=True,
            ),
        )
    ]

def build_classifier(model, callbacks, max_epochs, fold_seed=None, warm_start=False):
    train_generator = None
    if fold_seed is not None:
        train_generator = torch.Generator()
        train_generator.manual_seed(fold_seed)
    clf_kwargs = {
        "batch_size": CONFIG["batch_size"],
        "max_epochs": int(max_epochs),
        "device": DEVICE,
        "callbacks": callbacks,
        "train_split": make_train_split(),
        "classes": range(TARGET_N_CLASSES),
        "iterator_train__shuffle": True,
        "iterator_train__num_workers": 0,
        "iterator_valid__num_workers": 0,
        "optimizer": torch.optim.Adam,
        "warm_start": warm_start,
    }
    if CONFIG["learning_rate"] is not None:
        clf_kwargs["lr"] = CONFIG["learning_rate"]
    if train_generator is not None:
        clf_kwargs["iterator_train__generator"] = train_generator
    return EEGClassifier(model, **clf_kwargs)

def run_one_batch_finite_sanity_check(model, train_set, model_name):
    if len(train_set) == 0:
        raise RuntimeError(f"{model_name}: train_set is empty during sanity check.")
    batch_size = int(min(CONFIG["batch_size"], len(train_set)))
    sanity_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=False, num_workers=0)
    batch = next(iter(sanity_loader))
    x_batch = torch.as_tensor(batch[0]).float().to(DEVICE)
    y_batch = torch.as_tensor(batch[1]).long().to(DEVICE)
    was_training = model.training
    model = model.to(DEVICE)
    model.eval()
    with torch.no_grad():
        logits = model(x_batch)
        if not torch.isfinite(logits).all():
            raise RuntimeError(f"{model_name}: non-finite logits detected.")
        loss = torch.nn.functional.cross_entropy(logits, y_batch)
        if not torch.isfinite(loss):
            raise RuntimeError(f"{model_name}: non-finite loss detected.")
    if was_training:
        model.train()
    print(f"    Sanity check passed: finite logits/loss on one training batch for {model_name}.")

def extract_binary_score_vector(score_output, expected_n_samples):
    if score_output is None:
        return None
    scores = np.asarray(score_output)
    if scores.ndim == 1:
        score_vec = scores.astype(float)
    elif scores.ndim == 2 and scores.shape[1] == 2:
        score_vec = scores[:, 1].astype(float)
    elif scores.ndim == 2 and scores.shape[1] == 1:
        score_vec = scores[:, 0].astype(float)
    else:
        return None
    if score_vec.shape[0] != int(expected_n_samples) or not np.isfinite(score_vec).all():
        return None
    return score_vec

def compute_classification_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int).reshape(-1)
    y_pred = np.asarray(y_pred).astype(int).reshape(-1)
    metrics = {"accuracy": None, "balanced_accuracy": None}
    metrics["accuracy"] = float(accuracy_score(y_true, y_pred)) # type: ignore
    metrics["balanced_accuracy"] = float(balanced_accuracy_score(y_true, y_pred)) # type: ignore
    return metrics

def run_training_and_eval(train_set, test_set, fold_id, fold_label, n_total_folds=None):
    global PRETRAINED_CHECKPOINT_INFO
    if CONFIG["set_seed"]:
        fold_seed = BASE_SEED
        seed_everything(fold_seed)  # type: ignore
    else:
        fold_seed = None

    y_train = get_targets(train_set)
    y_test = get_targets(test_set)
    train_counts = np.bincount(y_train, minlength=TARGET_N_CLASSES)
    test_counts = np.bincount(y_test, minlength=TARGET_N_CLASSES)

    model_name = CONFIG["model_name"]
    strategy = CONFIG["strategy"]
    warmup_epochs = int(CONFIG["warmup_epochs"])
    model, pretrained_load_summary = initialize_model(model_name)
    PRETRAINED_CHECKPOINT_INFO = dict(pretrained_load_summary)

    total_folds_text = f"/{n_total_folds}" if n_total_folds is not None else ""
    print(f"\nFold {fold_id}{total_folds_text} | {fold_label}")
    print(f"    Train windows:           {len(train_set)} | counts={train_counts.tolist()}")
    print(f"    Test windows:            {len(test_set)} | counts={test_counts.tolist()}")
    print(f"    Downstream model:        {model_name}")
    print(f"    Fine-tune strategy:      {strategy}")
    print(f"    Pretrained loading path: {pretrained_load_summary['loading_path']}")
    print(f"    Pretrained repo:         {pretrained_load_summary.get('repo_id')}")

    if strategy == "new":
        phase_1_summary = set_trainable_params_for_phase(model, model_name, "new")
        assert_expected_trainable_scope(phase_1_summary, model_name, "new")
        print("    Phase 1 (new):")
        describe_trainable_params(phase_1_summary)
        run_one_batch_finite_sanity_check(model, train_set, model_name)
        clf = build_classifier(model, callbacks=make_callbacks(), max_epochs=int(CONFIG["n_epochs"]), fold_seed=fold_seed, warm_start=False)
        phase_summaries = {"phase_1": phase_1_summary, "phase_2": None}
        clf.fit(train_set, y=y_train)
    elif strategy == "full":
        if warmup_epochs < 1:
            raise ValueError("For strategy='full', warmup_epochs must be >= 1.")
        phase_1_summary = set_trainable_params_for_phase(model, model_name, "warmup")
        assert_expected_trainable_scope(phase_1_summary, model_name, "warmup")
        print("    Phase 1 (warmup):")
        describe_trainable_params(phase_1_summary)
        run_one_batch_finite_sanity_check(model, train_set, model_name)
        clf = build_classifier(model, callbacks=[], max_epochs=warmup_epochs, fold_seed=fold_seed, warm_start=True)
        clf.fit(train_set, y=y_train)
        phase_2_summary = set_trainable_params_for_phase(clf.module_, model_name, "full")
        print("    Phase 2 (full):")
        describe_trainable_params(phase_2_summary)
        clf.initialize_optimizer()
        remaining_epochs = int(CONFIG["n_epochs"]) - warmup_epochs
        if remaining_epochs < 1:
            raise ValueError("CONFIG['n_epochs'] must be greater than warmup_epochs for strategy='full'.")
        clf.set_params(callbacks=make_callbacks(), max_epochs=remaining_epochs)
        clf.fit(train_set, y=y_train)
        phase_summaries = {"phase_1": phase_1_summary, "phase_2": phase_2_summary}
    else:
        raise ValueError("CONFIG['strategy'] must be 'new' or 'full'.")

    y_pred = clf.predict(test_set)
    metrics = compute_classification_metrics(y_test, y_pred)

    stopped_epoch = int(clf.history[-1]["epoch"]) if len(clf.history) > 0 else 0  # type: ignore
    valid_loss_curve = [(int(row["epoch"]), float(row["valid_loss"])) for row in clf.history if "valid_loss" in row]
    if valid_loss_curve:
        best_epoch, best_valid_loss = min(valid_loss_curve, key=lambda x: x[1])
    else:
        best_epoch, best_valid_loss = None, None

    cm = confusion_matrix(y_test, y_pred, labels=np.arange(TARGET_N_CLASSES)).tolist()
    pred_hist = np.bincount(y_pred, minlength=TARGET_N_CLASSES).tolist()
    acc = metrics["accuracy"]
    bal_acc = metrics["balanced_accuracy"]
    print(f"    Result | best_epoch={best_epoch} | stop={stopped_epoch} | acc={acc:.4f} | bal_acc={bal_acc:.4f} | pred_hist={pred_hist}")

    return {
        "fold_id": int(fold_id),
        "fold_label": str(fold_label),
        "model_name": model_name,
        "strategy": strategy,
        "warmup_epochs": warmup_epochs,
        "n_train": len(train_set),
        "n_test": len(test_set),
        "train_class_counts": train_counts.tolist(),
        "test_class_counts": test_counts.tolist(),
        "pretrained_load": pretrained_load_summary,
        "phase_1_trainable_groups": phase_summaries["phase_1"]["trainable_groups"],
        "phase_1_trainable_params": phase_summaries["phase_1"]["trainable_params"],
        "phase_1_trainable_names": phase_summaries["phase_1"]["trainable_names"],
        "phase_2_trainable_groups": None if phase_summaries["phase_2"] is None else phase_summaries["phase_2"]["trainable_groups"],
        "phase_2_trainable_params": None if phase_summaries["phase_2"] is None else phase_summaries["phase_2"]["trainable_params"],
        "phase_2_trainable_names": None if phase_summaries["phase_2"] is None else phase_summaries["phase_2"]["trainable_names"],
        "best_epoch": best_epoch,
        "stopped_epoch": stopped_epoch,
        "best_valid_loss": best_valid_loss,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "confusion_matrix": cm,
        "prediction_histogram": pred_hist,
    }

## 5.2 Subject Cross-Validation Runner

In [ ]:
def make_fold_splits(y, n_folds, n_classes):
    indices = np.arange(len(y))
    counts = np.bincount(y, minlength=n_classes)
    min_class_count = counts.min()
    if min_class_count < n_folds:
        raise ValueError(f"Cannot use {n_folds} folds with class counts={counts.tolist()}. Reduce cv_folds or filter subjects.")
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=12)
    folds = []
    for fold_id, (train_idx, test_idx) in enumerate(skf.split(indices, y), start=1):
        folds.append({"fold_id": fold_id, "idx_train": train_idx, "idx_test": test_idx})
    return folds

def run_subject_cv(subject_id, subject_dataset, n_classes, cv_folds):
    y = get_targets(subject_dataset)
    counts = np.bincount(y, minlength=n_classes)
    print(f"\nSubject {subject_id}: {len(subject_dataset)} windows | class_counts={counts.tolist()}")
    folds = make_fold_splits(y, n_folds=cv_folds, n_classes=n_classes)
    results = []
    for fold in folds:
        train_set = Subset(subject_dataset, fold["idx_train"].tolist())
        test_set = Subset(subject_dataset, fold["idx_test"].tolist())
        fold_label = f"subject={subject_id}"
        result = run_training_and_eval(train_set, test_set, fold["fold_id"], fold_label, n_total_folds=cv_folds)
        result["subject_id"] = str(subject_id)
        results.append(result)

    acc_values = [r["accuracy"] for r in results if r["accuracy"] is not None]
    bal_acc_values = [r["balanced_accuracy"] for r in results if r["balanced_accuracy"] is not None]
    print(f"  Subject {subject_id} summary: acc={np.mean(acc_values):.4f}±{np.std(acc_values):.4f}  bal_acc={np.mean(bal_acc_values):.4f}±{np.std(bal_acc_values):.4f}")
    return results

## 5.3 Run All Subjects

In [ ]:
print("=" * 70)
print("STARTING WITHIN-SUBJECT CROSS-VALIDATION")
print("=" * 70)
print(f"Dataset:       {CONFIG['dataset_name']}")
print(f"Subjects:      {list(SUBJECT_WINDOWS.keys())}")
print(f"Model:         {CONFIG['model_name']}")
print(f"Pretrained:    {CONFIG['pretrained_mode']}")
print(f"Strategy:      {CONFIG['strategy']}")
print(f"CV folds:      {CONFIG['cv_folds']}")
print(f"Val split:     {CONFIG['val_split']}")
print(f"Max epochs:    {CONFIG['n_epochs']}")
print(f"Device:        {DEVICE}")
print("=" * 70)

FOLD_RESULTS = []
for sid, subject_ds in SUBJECT_WINDOWS.items():
    FOLD_RESULTS.extend(run_subject_cv(sid, subject_ds, TARGET_N_CLASSES, CONFIG["cv_folds"]))
print(f"\nTotal folds completed: {len(FOLD_RESULTS)}")

# 6. Results

## 6.1 Aggregate Metrics

In [ ]:
def aggregate_results(fold_results):
    # Per-subject aggregation when subject_id exists; per-heldout aggregation for LOSO.
    group_key = "subject_id" if any("subject_id" in r for r in fold_results) else "held_out_subject_id"
    grouped = {}
    for result in fold_results:
        gid = result.get(group_key, "global")
        grouped.setdefault(gid, {"accuracies": [], "balanced_accuracies": []})
        grouped[gid]["accuracies"].append(result.get("accuracy"))
        grouped[gid]["balanced_accuracies"].append(result.get("balanced_accuracy"))
    for gid, m in grouped.items():
        acc_values = [v for v in m["accuracies"] if v is not None]
        bal_values = [v for v in m["balanced_accuracies"] if v is not None]
        m["mean_accuracy"] = float(np.mean(acc_values)) if acc_values else None
        m["std_accuracy"] = float(np.std(acc_values)) if acc_values else None
        m["mean_balanced_accuracy"] = float(np.mean(bal_values)) if bal_values else None
        m["std_balanced_accuracy"] = float(np.std(bal_values)) if bal_values else None

    all_accs = [r["accuracy"] for r in fold_results if r.get("accuracy") is not None]
    all_bals = [r["balanced_accuracy"] for r in fold_results if r.get("balanced_accuracy") is not None]
    global_metrics = {
        "mean_accuracy": float(np.mean(all_accs)) if all_accs else None,
        "std_accuracy": float(np.std(all_accs)) if all_accs else None,
        "mean_balanced_accuracy": float(np.mean(all_bals)) if all_bals else None,
        "std_balanced_accuracy": float(np.std(all_bals)) if all_bals else None,
        "n_groups": len(grouped),
        "n_folds_total": len(fold_results),
    }
    return grouped, global_metrics

SUBJECT_METRICS, GLOBAL_METRICS = aggregate_results(FOLD_RESULTS)

print("=" * 70)
print("AGGREGATED RESULTS")
print("=" * 70)
for gid, m in sorted(SUBJECT_METRICS.items(), key=lambda x: _sort_subject_key(x[0])):
    acc_str = f"{m['mean_accuracy']:.4f}±{m['std_accuracy']:.4f}" if m["mean_accuracy"] is not None else "N/A"
    bal_str = f"{m['mean_balanced_accuracy']:.4f}±{m['std_balanced_accuracy']:.4f}" if m["mean_balanced_accuracy"] is not None else "N/A"
    print(f"  {gid}: acc={acc_str}  bal_acc={bal_str}")
print("-" * 70)
print(f"  OVERALL: acc={GLOBAL_METRICS['mean_accuracy']:.4f}±{GLOBAL_METRICS['std_accuracy']:.4f}  bal_acc={GLOBAL_METRICS['mean_balanced_accuracy']:.4f}±{GLOBAL_METRICS['std_balanced_accuracy']:.4f}")
print("=" * 70)

## 6.2 Experiment Summary

In [ ]:
print("\n" + "=" * 70)
print("EXPERIMENT SUMMARY")
print("=" * 70)
print(f"Run ID:                 {RUN_ID}")
print(f"Dataset:                {CONFIG['dataset_name']}")
print(f"Labels kept:            {CONFIG['labels_to_keep']}")
print(f"Model:                  {CONFIG['model_name']}")
print(f"Pretrained mode:        {CONFIG['pretrained_mode']}")
print(f"Strategy:               {CONFIG['strategy']}")
print(f"Window:                 {TARGET_TRIAL_DURATION_S:.3f}s / {WINDOW_SAMPLES} samples")
print(f"Artifacts:              {ARTIFACT_DIR}")
print(f"Mean Accuracy:          {GLOBAL_METRICS['mean_accuracy']:.4f} ± {GLOBAL_METRICS['std_accuracy']:.4f}")
print(f"Mean Balanced Accuracy: {GLOBAL_METRICS['mean_balanced_accuracy']:.4f} ± {GLOBAL_METRICS['std_balanced_accuracy']:.4f}")
print("=" * 70)

## 6.3 Save Artifacts

In [ ]:
cv_results_path = ARTIFACT_DIR / "cv_results.json"
with open(cv_results_path, "w") as f:
    json.dump(FOLD_RESULTS, f, indent=2)
subject_metrics_path = ARTIFACT_DIR / "subject_metrics.json"
with open(subject_metrics_path, "w") as f:
    json.dump(SUBJECT_METRICS, f, indent=2)
global_metrics_path = ARTIFACT_DIR / "global_metrics.json"
with open(global_metrics_path, "w") as f:
    json.dump(GLOBAL_METRICS, f, indent=2)

# Save window metadata separately because curated EEGMMIDB is not a Braindecode BaseConcatDataset.
window_metadata_path = ARTIFACT_DIR / "window_metadata.json"
window_metadata = {
    subject_id: arrays["metadata"]
    for subject_id, arrays in SUBJECT_ARRAYS.items()
}
with open(window_metadata_path, "w") as f:
    json.dump(window_metadata, f, indent=2)

run_metadata = {
    "run_id": RUN_ID,
    "artifact_dir": str(ARTIFACT_DIR),
    "dataset_name": CONFIG["dataset_name"],
    "dataset_source": CONFIG["dataset_source"],
    "dataset_path": str(DATASET_PATH),
    "preprocessed_dir": str(PREPROCESSED_DIR),
    "labels_to_keep": list(CONFIG["labels_to_keep"]),
    "excluded_subjects": list(CONFIG["exclude_subjects"]),
    "mi_runs_used": list(CONFIG.get("mi_runs_used", RUN_IDS)),
    "target_window_duration_s": TARGET_TRIAL_DURATION_S,
    "window_samples": WINDOW_SAMPLES,
    "subjects": [str(s) for s in SUBJECTS],
    "model_name": CONFIG["model_name"],
    "pretrained_mode": CONFIG["pretrained_mode"],
    "strategy": CONFIG["strategy"],
    "warmup_epochs": int(CONFIG["warmup_epochs"]),
    "pretrained_checkpoint_info": PRETRAINED_CHECKPOINT_INFO,
    "cv_seed": BASE_SEED,
    "global_metrics": GLOBAL_METRICS,
}
run_metadata_path = ARTIFACT_DIR / "run_metadata.json"
with open(run_metadata_path, "w") as f:
    json.dump(run_metadata, f, indent=2)

print(f"CV results saved to:      {cv_results_path}")
print(f"Subject metrics saved to: {subject_metrics_path}")
print(f"Global metrics saved to:  {global_metrics_path}")
print(f"Window metadata saved to: {window_metadata_path}")
print(f"Run metadata saved to:    {run_metadata_path}")
print(f"\nAll artifacts in: {ARTIFACT_DIR}")